# 1차시 실습 — 랜덤 점심 추천기

**프롬프트만으로 화면을 만들고, 작게 반복하며 키운다 (iteration loop 체감)**

## 🧪 이 노트북 사용법

- 이 노트북은 워크샵 **실습(Codex CLI)의 실행 가능한 동반 자료**입니다. 각자 발급받은 **`OPENAI_API_KEY`** 로 바로 돌아갑니다.
- 실제 캠프에서는 **터미널의 Codex CLI**로 같은 작업을 합니다. 그래서 각 단계마다
  - **▶︎ 노트북에서 실행** (파이썬 셀) 과
  - **⌨️ 터미널 Codex 명령** (복붙용) 을 함께 적어 두었습니다.
- 위에서부터 **순서대로** 셀을 실행하세요 (`Shift+Enter`).
- 황금률: **작게 요청 → 확인 → 수정** (1차시 iteration loop).

> ⚠️ API 호출에는 약간의 토큰 비용이 듭니다. 기본 모델은 저렴한 `gpt-4o-mini`.

In [1]:
# ✅ 환경 준비 — 맨 처음 한 번만 실행
%pip install -q openai
import os, getpass, re, pathlib, json
from openai import OpenAI
from IPython.display import IFrame, Markdown, display

# 각자 발급받은 OPENAI_API_KEY 입력 (입력값은 화면에 보이지 않습니다)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY를 붙여넣고 Enter: ")

client = OpenAI()
MODEL = "gpt-4o-mini"   # 품질을 더 원하면 "gpt-4o" 또는 "gpt-4.1" 로 바꾸세요

def ask(prompt, system=None, **kw):
    """프롬프트 하나를 보내고 답변 텍스트를 돌려준다."""
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(model=MODEL, messages=msgs, **kw)
    return r.choices[0].message.content

def show(text):
    display(Markdown(text))

print("준비 완료! MODEL =", MODEL)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
준비 완료! MODEL = gpt-4o-mini


In [2]:
# 🔧 HTML 생성·미리보기 도우미 (HTML을 만드는 차시에서 사용)
def extract_code(text):
    """답변에서 첫 번째 코드블록만 꺼낸다. 없으면 원문 그대로."""
    if "```" not in text:
        return text.strip()
    block = text.split("```", 2)[1]
    if "\n" in block:
        first, rest = block.split("\n", 1)
        if first.strip().isalpha():   # ```html 같은 언어 태그 제거
            return rest.strip()
    return block.strip()

def save_and_preview(html, filename="app.html", height=420):
    pathlib.Path(filename).write_text(html, encoding="utf-8")
    print("저장됨 →", filename, " (브라우저로 직접 열어도 됩니다)")
    return IFrame(filename, width="100%", height=height)

def iterate(current_html, instruction):
    """현재 HTML 전체를 보여주고(=컨텍스트) 한 가지 수정을 요청한다."""
    prompt = f"""아래는 현재 HTML 전체야:
```html
{current_html}
```
다음 한 가지만 반영해서 **전체 HTML을 다시** 출력해줘: {instruction}
- 설명 없이 HTML 코드만 출력해."""
    return extract_code(ask(prompt))

## 🌈 바이브 코딩으로 뭘 만들 수 있나 — 실제 예시 (출처 포함)

오늘은 작은 점심 추천기지만, **같은 방식(작게 요청 → 확인 → 수정)** 으로 이런 것들이 실제로 만들어졌습니다. (괄호 = 만든 도구·사이트)

**개인·취미**
- **Refetch** — 해커뉴스 대안 뉴스 사이트. 아이디어→완성 약 **15시간**(넷플릭스 보며). 〔Appwrite · appwrite.io〕
- **Storypot** — 이모지를 냄비에 넣으면 짧은 **동화**로. 아빠가 아이 위해 주말에. 〔Replit · replit.com〕
- **Timeless Memories** — 옛 가족사진을 **AI 영상**으로 / **Lambo Levels** — 크립토 수익 **시각화**. 〔Lovable · lovable.dev〕

**제품·수익·MVP**
- **Disko** — SMS 마케팅 서비스, 4개월 만에 **월매출 $11k**. 〔Replit · replit.com〕
- **Xenon** — 비디오 에디터 MVP를 **2주**에 / **Potluck AI** — 재료·기분 기반 레시피 추천. 〔Lovable(+Supabase·Gemini)〕
- **Boltception** — 프롬프트로 웹사이트 만드는 빌더 〔bolt.new〕 / **Airbnb 클론**을 **17분**에 〔GPT-5 + Lovable〕

**종류도 다양** — 웹앱 · 게임 · 크롬 확장 · 대시보드 · 자동화 · 봇 · 3D 사이트 · 리서치 툴. 〔awesome-vibe-coding, Questera〕

**대표 빌드 도구(사이트)**: Lovable(lovable.dev) · Bolt(bolt.new) · Replit(replit.com) · v0(v0.dev) · Cursor(cursor.com) · Appwrite(appwrite.io) · Supabase(supabase.com)
> 우리 캠프는 이 중 **터미널 Codex CLI**로 진행합니다 — 도구는 달라도 *작게 요청→확인→수정* 원리는 같습니다.

> ⚠️ 단, 바이브 코딩은 **프로토타입**에 특히 강합니다. 실서비스로 내보내기 전엔 **사람이 코드를 검토**하세요.

**출처**: [Zapier 사례](https://zapier.com/blog/vibe-coding-examples/) · [Appwrite 사례](https://appwrite.io/blog/post/examples-of-vibe-coding) · [Agen.cy 20+ 프로젝트 모음](https://medium.com/@agencyai/from-mvps-to-mobile-apps-20-real-projects-built-with-v0-lovable-bolt-replit-ed0185cb2821) · [Lovable 가이드](https://lovable.dev/guides/best-vibe-coding-tools-2026-build-apps-chatting) · [awesome-vibe-coding](https://github.com/filipecalegario/awesome-vibe-coding) · [Google Cloud: vibe coding](https://cloud.google.com/discover/what-is-vibe-coding)

## 🎯 오늘 만들 것 — 스펙 (S78)

- 입력 없음 / **버튼 1개** / 클릭 시 메뉴 1개 표시 / **단일 HTML 파일**
- 도구는 **하나** (캠프에선 Codex CLI). v0·스티치 같은 별도 디자인 도구 없음.
- 핵심 학습 포인트는 결과물이 아니라 **루프 감각**: 작게 요청 → 확인 → 수정.

## 1단계 — 한 프롬프트로 생성 (S79)

⌨️ **터미널 Codex:** `codex "버튼을 누르면 랜덤으로 점심 메뉴를 추천하는 단일 HTML 페이지를 만들어줘"`

▶︎ 아래 셀을 실행하면 같은 요청을 API로 보내 HTML을 생성하고 바로 미리봅니다.

In [3]:
prompt1 = "버튼을 누르면 랜덤으로 점심 메뉴를 추천하는 단일 HTML 페이지를 만들어줘. 한국어로. 설명 없이 HTML 코드만."
html = extract_code(ask(prompt1))
save_and_preview(html, "app.html")

저장됨 → app.html  (브라우저로 직접 열어도 됩니다)


## 2단계 — iterate: 필터 추가 (S80)

작게 요청한다 — 한 번에 하나만. 이전 결과(HTML 전체)를 **컨텍스트로 함께** 넘기는 게 포인트(→ 2차시 컨텍스트 엔지니어링).

⌨️ **터미널 Codex:** `codex "한식/양식/중식 필터 버튼을 추가해줘"`

In [4]:
html = iterate(html, "한식/양식/중식 필터 버튼을 추가하고, 선택한 종류 안에서만 랜덤 추천해줘.")
save_and_preview(html, "app.html")

저장됨 → app.html  (브라우저로 직접 열어도 됩니다)


## 3단계 — iterate: 최근 기록

⌨️ **터미널 Codex:** `codex "최근 추천 5개를 아래에 기록해줘"`

In [5]:
html = iterate(html, "최근 추천한 메뉴 5개를 화면 아래에 목록으로 기록해서 보여줘.")
save_and_preview(html, "app.html")

저장됨 → app.html  (브라우저로 직접 열어도 됩니다)


## 🆘 막히면 

1. 에러/이상한 결과를 **그대로** 다시 붙여넣기
2. **한 단계만** 다시 요청 (욕심내지 않기)
3. 요청을 **더 작게** 쪼개기

```python
# 직접 한 줄씩 바꿔가며 더 실험해 보세요
html = iterate(html, "여기에 원하는 수정을 한 가지만 적기")
save_and_preview(html, "app.html")
```

## 🔁 학습 포인트

- 한 번에 완성하지 않는다 — **작게·자주·반복**.
- 이 앱을 **2차시**에서 규칙·리뷰·버전관리를 갖춘 *정식 프로젝트*로 승격합니다. `app.html`을 남겨 두세요.